# Lahore Environment + Urban Form Index

This notebook builds grid-based environmental (EQ) and urban development (UD) indices for Lahore and exports maps.


In [2]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import fiona
from shapely.ops import nearest_points
try:
    from scipy.spatial import cKDTree as KDTree
except Exception:
    KDTree = None

try:
	ee.Initialize()
except Exception:
	
	ee.Authenticate()
	ee.Initialize()

/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Inputs

Define study area and load data sources (LST, NDVI, NL, AQI, building metrics, OSM roads).


In [3]:
UC_SHP = "../data/Lahore UCs/Lahore UC.shp"
# If UC shapefile is missing, you can swap to data/Union_Councils.geojson
# UC_SHP = "../data/Union_Councils.geojson"

gdf = gpd.read_file(UC_SHP).to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()
# Lahore boundary for clipping (WGS84 here; reproject later)
lahore_boundary = gdf.unary_union
YEAR, MONTH = 2024, 10


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_10576/932579517.py:10: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lahore_boundary = gdf.unary_union


In [4]:
LST = "LST/LST_Lahore_points_100m.geojson"
NDVI = "NDVI/NDVI_Lahore_Oct2024_points_100m.geojson"
NL = "NL/VIIRS_NL_Lahore_Oct2024_points_250m.geojson"
BFP_HR = "Building+Highrise/OpenBuildings_Lahore_2023.geojson"
AQI_UC = "AQI/AQI_2024_10_UC_interpolated.geojson"
POI_PATH = "POI/data/pois_lahore.geojson"
POI_ACCESS_PATH_PRIMARY = "POI/lahore_poi_access_heatmap.geojson"
OSM_ROADS_GPKG = "OSM/lahore_roads.gpkg"
OSM_ROADS_LAYER = None

def load_points_geometry(path, value_field):
    gdf = gpd.read_file(path).to_crs(4326)
    fc = geemap.gdf_to_ee(gdf)
    return fc

def load_points_geojson(path, value_field):
    gdf = gpd.read_file(path).to_crs(4326)
    return gdf

lst = load_points_geojson(LST, "LST_Day_1km")
ndvi = load_points_geojson(NDVI, "NDVI")
nl = load_points_geojson(NL, "avg_rad")
bfp_hr = load_points_geojson(BFP_HR, "highrise_density")

try:
    aqi_uc = gpd.read_file(AQI_UC).to_crs(4326)
except Exception as exc:
    print(f"AQI load failed: {exc}")
    aqi_uc = gpd.GeoDataFrame(columns=["AQI_mean", "geometry"], geometry="geometry", crs="EPSG:4326")

try:
    if OSM_ROADS_LAYER is None:
        layers = fiona.listlayers(OSM_ROADS_GPKG)
        layer = layers[0] if layers else None
    else:
        layer = OSM_ROADS_LAYER
    roads = gpd.read_file(OSM_ROADS_GPKG, layer=layer).to_crs(4326)
except Exception as exc:
    print(f"Roads load failed: {exc}")
    roads = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

try:
    pois = gpd.read_file(POI_PATH).to_crs(4326)
except Exception as exc:
    print(f"POIs load failed: {exc}")
    pois = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")


## Quick Sanity Checks


In [5]:
print(lst.shape, lst.columns)
print(ndvi.shape, ndvi.columns)
print(nl.shape, nl.columns)
print(bfp_hr.shape, bfp_hr.columns)

(210220, 5) Index(['id', 'val', 'longitude', 'latitude', 'geometry'], dtype='object')
(210461, 5) Index(['id', 'val', 'longitude', 'latitude', 'geometry'], dtype='object')
(33678, 5) Index(['id', 'val', 'longitude', 'latitude', 'geometry'], dtype='object')
(841968, 7) Index(['id', 'height', 'highrise', 'latitude', 'longitude', 'presence',
       'geometry'],
      dtype='object')


## Grid + Projection Setup


In [5]:
GRID_SIZE_M_100 = 100
GRID_SIZE_M_250 = 250
CRS_METRIC = "EPSG:32643"   # UTM zone 43N fits Lahore

# ==== 1) Reproject to metric CRS ====
ndvi_m   = ndvi.to_crs(CRS_METRIC)
lst_m    = lst.to_crs(CRS_METRIC)
nl_m     = nl.to_crs(CRS_METRIC)
bfp_m    = bfp_hr.to_crs(CRS_METRIC)

if not aqi_uc.empty:
    aqi_m = aqi_uc.to_crs(CRS_METRIC)
else:
    aqi_m = aqi_uc

if not roads.empty:
    roads_m = roads.to_crs(CRS_METRIC)
else:
    roads_m = roads

lahore_boundary_m = gpd.GeoSeries([lahore_boundary], crs=4326).to_crs(CRS_METRIC).iloc[0]


## Grid Construction


In [6]:
from shapely.geometry import box

# ==== 2) Build grid + clip to Lahore boundary ====
def extent_union(gdfs):
    xmin = min(g.total_bounds[0] for g in gdfs)
    ymin = min(g.total_bounds[1] for g in gdfs)
    xmax = max(g.total_bounds[2] for g in gdfs)
    ymax = max(g.total_bounds[3] for g in gdfs)
    return xmin, ymin, xmax, ymax

def make_grid_from_gdfs(gdfs, step):
    xmin, ymin, xmax, ymax = extent_union(gdfs)
    x0 = (np.floor(xmin / step) * step)
    y0 = (np.floor(ymin / step) * step)
    x1 = (np.ceil(xmax / step) * step)
    y1 = (np.ceil(ymax / step) * step)
    xs = np.arange(x0, x1, step, dtype=float)
    ys = np.arange(y0, y1, step, dtype=float)

    polys, rows = [], []
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            polys.append(box(x, y, x + step, y + step))
            rows.append((i, j))

    g = gpd.GeoDataFrame(rows, columns=["gx", "gy"], geometry=polys, crs=CRS_METRIC)
    gx = g["gx"].astype(np.int64)
    gy = g["gy"].astype(np.int64)
    g["cell_id"] = (np.left_shift(gx, 32) + gy).astype(np.int64)
    return g

def clip_grid(grid, boundary_geom):
    # Clip to Lahore to avoid empty cells outside study area
    clipped = grid[grid.intersects(boundary_geom)].copy()
    clipped.reset_index(drop=True, inplace=True)
    return clipped

# Build and clip grids
grid_250 = clip_grid(make_grid_from_gdfs([ndvi_m, lst_m, nl_m], GRID_SIZE_M_250), lahore_boundary_m)
grid_100 = clip_grid(make_grid_from_gdfs([ndvi_m, lst_m], GRID_SIZE_M_100), lahore_boundary_m)

for grid, step in [(grid_250, GRID_SIZE_M_250), (grid_100, GRID_SIZE_M_100)]:
    grid["area_m2"] = step * step
    grid["area_ha"] = grid["area_m2"] / 10_000.0


## Aggregation Helpers


In [7]:
# ==== 3) Spatial-join points to grid and aggregate ====
def agg_mean(gdf_points, value_col, grid):
    cols = [value_col, "geometry"]
    j = gpd.sjoin(gdf_points[cols], grid[["cell_id", "geometry"]], how="inner", predicate="intersects")
    return j.groupby("cell_id", as_index=False)[value_col].mean()

def agg_count(gdf_points, grid):
    j = gpd.sjoin(gdf_points[["geometry"]], grid[["cell_id", "geometry"]], how="inner", predicate="intersects")
    return j.groupby("cell_id", as_index=False).size().rename(columns={"size": "count"})

def agg_area_weighted(polygons, value_col, grid):
    if polygons.empty:
        return gpd.GeoDataFrame(columns=["cell_id", value_col])
    inter = gpd.overlay(grid[["cell_id", "geometry"]], polygons[[value_col, "geometry"]], how="intersection")
    inter["area_m2"] = inter.geometry.area
    inter["weighted"] = inter[value_col] * inter["area_m2"]
    out = inter.groupby("cell_id", as_index=False).agg({"weighted": "sum", "area_m2": "sum"})
    out[value_col] = out["weighted"] / out["area_m2"]
    return out[["cell_id", value_col]]

def agg_line_length(lines, grid):
    if lines.empty:
        return gpd.GeoDataFrame(columns=["cell_id", "len_m"])
    inter = gpd.overlay(lines[["geometry"]], grid[["cell_id", "geometry"]], how="intersection")
    inter["len_m"] = inter.geometry.length
    return inter.groupby("cell_id", as_index=False)["len_m"].sum()

def agg_highrise_decay(points, grid, radius_m=500):
    # Distance-decay from nearest highrise point to each grid cell centroid
    if points.empty:
        return gpd.GeoDataFrame(columns=["cell_id", "Highrise_decay"])

    centroids = grid[["cell_id", "geometry"]].copy()
    centroids["geometry"] = centroids.geometry.centroid

    try:
        joined = gpd.sjoin_nearest(centroids, points[["geometry"]], how="left", distance_col="dist_m")
    except Exception:
        # Fallback: if sjoin_nearest unavailable, return empty decay
        return gpd.GeoDataFrame(columns=["cell_id", "Highrise_decay"])

    joined["Highrise_decay"] = np.exp(-(joined["dist_m"] / radius_m))
    return joined[["cell_id", "Highrise_decay"]]



## POI Accessibility

Compute road-network travel time (minutes) from each grid cell to the nearest POI using the **new** Lahore roads file.


In [9]:
# ---- POI ACCESS FROM PRECOMPUTED GRID ----
# Uses OSM/lahore_poi_access_heatmap.geojson, then fallback to POI/lahore_poi_access_heatmap.geojson

def compute_poi_access(grid):
    # Lazy-load, so this works even if earlier data-load cell was not run in order.
    poi_access_gdf = globals().get("poi_access", None)
    if poi_access_gdf is None:
        loaded = False
        for path in [POI_ACCESS_PATH_PRIMARY]:
            try:
                poi_access_gdf = gpd.read_file(path).to_crs(4326)
                print(f"Loaded POI access from: {path}")
                loaded = True
                break
            except Exception:
                continue
        if not loaded:
            poi_access_gdf = gpd.GeoDataFrame(columns=["n1_min", "geometry"], geometry="geometry", crs="EPSG:4326")
        globals()["poi_access"] = poi_access_gdf

    if poi_access_gdf.empty:
        return pd.DataFrame({"cell_id": grid["cell_id"], "POI_n1_min": np.nan})

    centroids = grid.copy()
    centroids["geometry"] = centroids.geometry.centroid
    centroids = centroids.set_crs(CRS_METRIC).to_crs(4326)

    joined = gpd.sjoin_nearest(
        centroids,
        poi_access_gdf[["n1_min", "geometry"]],
        how="left",
        distance_col="dist_m",
    )
    out = joined[["cell_id", "n1_min"]].rename(columns={"n1_min": "POI_n1_min"})
    return out


## Index Construction


In [10]:
# ==== 4) Build indices for 250m (full) and 100m (no NL) ====
def robust_0_10(series, invert=False, qlo=0.05, qhi=0.95):
    s = series.astype(float)
    lo, hi = s.quantile(qlo), s.quantile(qhi)
    denom = max(hi - lo, 1e-9)
    z = ((s - lo) / denom).clip(0, 1) * 10.0
    return (10 - z) if invert else z

def weighted_average(frame, cols, weights):
    w = np.array([weights[c] for c in cols], dtype=float)
    vals = frame[cols].values
    mask = np.isnan(vals)
    w_masked = np.where(mask, 0.0, w)
    denom = w_masked.sum(axis=1)
    out = np.where(denom == 0, np.nan, np.nansum(vals * w, axis=1) / denom)
    return out

def build_index(grid, include_nl=True):
    # NDVI/LST/NL averages
    ndvi_mean = agg_mean(ndvi_m, "val", grid).rename(columns={"val": "NDVI"})
    lst_mean  = agg_mean(lst_m,  "val", grid).rename(columns={"val": "LST"})
    if include_nl:
        nl_mean = agg_mean(nl_m, "val", grid).rename(columns={"val": "NL"})
    else:
        nl_mean = None

    # AQI (area-weighted from UC polygons)
    aqi_mean = agg_area_weighted(aqi_m, "AQI_mean", grid).rename(columns={"AQI_mean": "AQI"})

    # Building metrics
    bfp_presence = agg_mean(bfp_m, "presence", grid).rename(columns={"presence": "BFP_presence"})
    bfp_highrise = agg_mean(bfp_m, "highrise", grid).rename(columns={"highrise": "Highrise_share"})
    bfp_height   = agg_mean(bfp_m, "height", grid).rename(columns={"height": "Height_avg"})
    bfp_count    = agg_count(bfp_m, grid).rename(columns={"count": "BFP_count"})

    highrise_pts = bfp_m[bfp_m["highrise"] > 0] if "highrise" in bfp_m.columns else bfp_m.iloc[0:0]
    highrise_decay = agg_highrise_decay(highrise_pts, grid, radius_m=500)

    # Road length (meters) within grid cell
    road_len = agg_line_length(roads_m, grid).rename(columns={"len_m": "Road_len_m"})

    g = grid[["cell_id", "geometry", "area_m2", "area_ha"]].copy()
    g = g.merge(ndvi_mean, on="cell_id", how="left")
    g = g.merge(lst_mean,  on="cell_id", how="left")
    if include_nl and nl_mean is not None:
        g = g.merge(nl_mean, on="cell_id", how="left")
    g = g.merge(aqi_mean, on="cell_id", how="left")
    g = g.merge(bfp_presence, on="cell_id", how="left")
    g = g.merge(bfp_highrise, on="cell_id", how="left")
    g = g.merge(bfp_height, on="cell_id", how="left")
    g = g.merge(bfp_count, on="cell_id", how="left")
    g = g.merge(road_len, on="cell_id", how="left")
    g = g.merge(highrise_decay, on="cell_id", how="left")

    # POI accessibility (minutes to nearest POI)
    poi_access = compute_poi_access(grid)
    g = g.merge(poi_access, on="cell_id", how="left")

    g["BFP_density"] = g["BFP_count"] / g["area_ha"]
    g["Road_density"] = g["Road_len_m"] / g["area_ha"]

    # ==== 5) Robust 0–10 scaling (5–95%), invert where lower-is-better ====
    g["NDVI_sc"] = robust_0_10(g["NDVI"], invert=False)
    g["LST_sc"]  = robust_0_10(g["LST"],  invert=True)
    if include_nl:
        g["NL_sc"] = robust_0_10(g["NL"], invert=True)
    g["AQI_sc"] = robust_0_10(g["AQI"], invert=True)

    g["BFP_presence_sc"] = robust_0_10(g["BFP_presence"], invert=False)
    g["Highrise_sc"]     = robust_0_10(g["Highrise_share"], invert=False)
    g["Height_sc"]       = robust_0_10(g["Height_avg"], invert=False)
    g["BFP_density_sc"]  = robust_0_10(g["BFP_density"], invert=True)
    g["Road_density_sc"] = robust_0_10(g["Road_density"], invert=True)
    g["Highrise_decay_sc"] = robust_0_10(g["Highrise_decay"], invert=False)
    g["POI_access_sc"] = robust_0_10(g["POI_n1_min"], invert=True)

    # ==== 6) Compose indices (weighted) ====
    if include_nl:
        EQ_cols = ["NDVI_sc", "LST_sc", "NL_sc", "AQI_sc", "POI_access_sc"]
        EQ_weights = {"NDVI_sc": 0.4, "LST_sc": 0.2, "NL_sc": 0.10, "AQI_sc": 0.2, "POI_access_sc": 0.1}
    else:
        EQ_cols = ["NDVI_sc", "LST_sc", "AQI_sc", "POI_access_sc"]
        EQ_weights = {"NDVI_sc": 0.35, "LST_sc": 0.35, "AQI_sc": 0.2, "POI_access_sc": 0.1}

    UD_cols = ["BFP_presence_sc", "Highrise_decay_sc", "Height_sc", "BFP_density_sc", "Road_density_sc"]
    UD_weights = {
        "BFP_presence_sc": 0.1,
        "Highrise_decay_sc": 0.3,
        "Height_sc": 0.2,
        "BFP_density_sc": 0.2,
        "Road_density_sc": 0.2,
    }

    g["EQ"] = weighted_average(g, EQ_cols, EQ_weights)
    g["UD"] = weighted_average(g, UD_cols, UD_weights)

    OVERALL_cols = ["EQ", "UD"]
    OVERALL_weights = {"EQ": 0.4, "UD": 0.6}
    g["Overall"] = weighted_average(g, OVERALL_cols, OVERALL_weights)
    g["Overall_vis"] = robust_0_10(g["Overall"], qlo=0.1, qhi=0.9)

    return g

out_250 = build_index(grid_250, include_nl=True)
out_100 = build_index(grid_100, include_nl=False)

# ==== 7) Save outputs ====
out_250_wgs = out_250.to_crs(4326)
out_100_wgs = out_100.to_crs(4326)

out_250_wgs[[
    "cell_id", "NDVI", "LST", "NL", "AQI", "BFP_presence", "Highrise_share", "Highrise_decay", "Height_avg", "BFP_count", "BFP_density",
    "Road_len_m", "Road_density", "POI_n1_min",
    "NDVI_sc", "LST_sc", "NL_sc", "AQI_sc", "BFP_presence_sc", "Highrise_decay_sc", "Height_sc", "BFP_density_sc", "Road_density_sc",
    "EQ", "UD", "Overall", "Overall_vis", "geometry"
]].to_file("lahore_index_250m.geojson", driver="GeoJSON")

out_100_wgs[[
    "cell_id", "NDVI", "LST", "AQI", "BFP_presence", "Highrise_share", "Highrise_decay", "Height_avg", "BFP_count", "BFP_density",
    "Road_len_m", "Road_density", "POI_n1_min",
    "NDVI_sc", "LST_sc", "AQI_sc", "BFP_presence_sc", "Highrise_decay_sc", "Height_sc", "BFP_density_sc", "Road_density_sc",
    "EQ", "UD", "Overall", "Overall_vis", "geometry"
]].to_file("lahore_index_100m.geojson", driver="GeoJSON")

out_250_wgs.drop(columns="geometry").to_csv("lahore_index_250m.csv", index=False)
out_100_wgs.drop(columns="geometry").to_csv("lahore_index_100m.csv", index=False)

print("Saved: lahore_index_250m.geojson/.csv, lahore_index_100m.geojson/.csv")
print("250m:", out_250_wgs[["EQ", "UD", "Overall"]].describe())
print("100m:", out_100_wgs[["EQ", "UD", "Overall"]].describe())


Loaded POI access from: POI/lahore_poi_access_heatmap.geojson


/Users/ahmed/Library/Python/3.9/lib/python/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/Users/ahmed/Library/Python/3.9/lib/python/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Saved: lahore_index_250m.geojson/.csv, lahore_index_100m.geojson/.csv
250m:                  EQ            UD       Overall
count  29214.000000  29214.000000  29214.000000
mean       4.629586      2.152114      3.143103
std        2.153804      1.741292      1.046672
min        0.000000      0.000000      0.000000
25%        3.015314      0.501226      2.445620
50%        4.467042      1.961157      3.104983
75%        6.045154      3.208013      3.823363
max       10.000000      9.696295      9.453143
100m:                   EQ             UD        Overall
count  180384.000000  180384.000000  180384.000000
mean        4.187123       3.004630       3.477628
std         2.238024       1.970663       1.235832
min         0.000000       0.000000       0.000000
25%         2.453081       2.129571       2.644371
50%         3.908859       2.521944       3.483968
75%         5.670376       4.001560       4.304547
max        10.000000      10.000000       9.787039


## Maps


In [ ]:
import folium

# Final maps (250m and 100m)
center_250 = out_250_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center_250.y, center_250.x], zoom_start=11, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=out_250_wgs.to_json(),
    data=out_250_wgs,
    columns=["cell_id", "Overall_vis"],
    key_on="feature.properties.cell_id",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.1,
    name="Overall (250m)",
    legend_name="Overall Index 250m (visual scaled)",
).add_to(m)

folium.Choropleth(
    geo_data=out_100_wgs.to_json(),
    data=out_100_wgs,
    columns=["cell_id", "Overall_vis"],
    key_on="feature.properties.cell_id",
    fill_color="YlGnBu",
    fill_opacity=0.6,
    line_opacity=0.1,
    name="Overall (100m)",
    legend_name="Overall Index 100m (visual scaled)",
).add_to(m)

folium.LayerControl().add_to(m)

m.save("lahore_index_map_final.html")
m


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_91171/1455963785.py:4: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center_250 = out_250_wgs.geometry.unary_union.centroid


## Ablations

These recompute the overall index by removing each component and save maps + GeoJSONs for comparison.


In [1]:
# ==== ABLATIONS: recompute Overall without each component and save PNG maps ====
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd

ABL_DIR = "ablations"
os.makedirs(ABL_DIR, exist_ok=True)

# Base weights (must match build_index)
EQ_weights_250 = {"NDVI_sc": 0.4, "LST_sc": 0.2, "NL_sc": 0.10, "AQI_sc": 0.2, "POI_access_sc": 0.1}
EQ_weights_100 = {"NDVI_sc": 0.35, "LST_sc": 0.35, "AQI_sc": 0.2, "POI_access_sc": 0.1}

UD_weights = {
    "BFP_presence_sc": 0.1,
    "Highrise_decay_sc": 0.3,
    "Height_sc": 0.2,
    "BFP_density_sc": 0.2,
    "Road_density_sc": 0.2,
}

OVERALL_weights = {"EQ": 0.4, "UD": 0.6}


# Standalone fallbacks so ablations run even if prior cells were skipped
def weighted_average(frame, cols, weights):
    w = np.array([weights[k] for k in cols], dtype=float)
    vals = frame[cols].values
    mask = np.isnan(vals)
    w_masked = np.where(mask, 0.0, w)
    denom = w_masked.sum(axis=1)
    out = np.where(denom == 0, np.nan, np.nansum(vals * w, axis=1) / denom)
    return out

def robust_0_10(series, invert=False, qlo=0.05, qhi=0.95):
    s = pd.Series(series).astype(float)
    lo, hi = s.quantile(qlo), s.quantile(qhi)
    denom = max(hi - lo, 1e-9)
    z = ((s - lo) / denom).clip(0, 1) * 10.0
    return (10 - z) if invert else z


def renormalize(weights, drop_key=None):
    w = {k: v for k, v in weights.items() if k != drop_key}
    s = sum(w.values())
    if s == 0:
        return w
    return {k: v / s for k, v in w.items()}


def compute_ablation(g, include_nl, drop_eq=None, drop_ud=None, drop_overall=None):
    eq_w_full = EQ_weights_250 if include_nl else EQ_weights_100

    # Keep only columns that exist in provided data
    eq_w = {k: v for k, v in eq_w_full.items() if k in g.columns}
    ud_w = {k: v for k, v in UD_weights.items() if k in g.columns}

    if drop_eq and drop_eq in eq_w:
        eq_w = renormalize(eq_w, drop_eq)
    if drop_ud and drop_ud in ud_w:
        ud_w = renormalize(ud_w, drop_ud)

    EQ = weighted_average(g, list(eq_w.keys()), eq_w) if len(eq_w) else np.full(len(g), np.nan)
    UD = weighted_average(g, list(ud_w.keys()), ud_w) if len(ud_w) else np.full(len(g), np.nan)

    if drop_overall == "EQ":
        Overall = UD
    elif drop_overall == "UD":
        Overall = EQ
    else:
        Overall = weighted_average(pd.DataFrame({"EQ": EQ, "UD": UD}), ["EQ", "UD"], OVERALL_weights)

    Overall_vis = robust_0_10(pd.Series(Overall), qlo=0.1, qhi=0.9)
    return EQ, UD, Overall, Overall_vis



def save_map_png(g_wgs, col, name, eq_w, ud_w, overall_w, cmap="RdYlGn"):
    fig, ax = plt.subplots(figsize=(10, 10))
    g_wgs.plot(column=col, ax=ax, cmap=cmap, legend=True, linewidth=0.05, edgecolor="none", legend_kwds={"shrink": 0.65, "fraction": 0.03, "pad": 0.01})
    ax.set_title(f"{name} ({col})")
    ax.set_axis_off()
    def fmt_weights(title, d):
        items = [f"{k.replace('_sc','')}: {v*100:.0f}%" for k, v in d.items()]
        return f"{title}\n  " + "\n  ".join(items)

    weights_text = "\n\n".join([
        fmt_weights("EQ Weights", eq_w),
        fmt_weights("UD Weights", ud_w),
        fmt_weights("Overall Weights", overall_w),
    ])

    fig.text(
        0.015,
        0.02,
        weights_text,
        ha="left",
        va="bottom",
        fontsize=8,
        family="DejaVu Sans",
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#666", "alpha": 0.9},
    )
    out_png = os.path.join(ABL_DIR, f"{name}.png")
    fig.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.close(fig)


def run_ablations(out_wgs, include_nl, tag):
    g_work = out_wgs.copy()

    # Backward-compat: old saved index files may have POI_n1_min but not POI_access_sc
    if "POI_access_sc" not in g_work.columns and "POI_n1_min" in g_work.columns:
        g_work["POI_access_sc"] = robust_0_10(g_work["POI_n1_min"], invert=True)

    specs = [("drop_EQ", {"drop_overall": "EQ"}), ("drop_UD", {"drop_overall": "UD"})]

    eq_weights = EQ_weights_250 if include_nl else EQ_weights_100
    eq_keys = [k for k in eq_weights.keys() if k in g_work.columns]
    ud_keys = [k for k in UD_weights.keys() if k in g_work.columns]

    missing_eq = [k for k in eq_weights.keys() if k not in g_work.columns]
    missing_ud = [k for k in UD_weights.keys() if k not in g_work.columns]
    if missing_eq:
        print(f"[info] Missing EQ metrics for {tag}: {missing_eq}")
    if missing_ud:
        print(f"[info] Missing UD metrics for {tag}: {missing_ud}")

    for k in eq_keys:
        specs.append((f"drop_{k}", {"drop_eq": k}))

    for k in ud_keys:
        specs.append((f"drop_{k}", {"drop_ud": k}))

    for name, opts in specs:
        EQ, UD, Overall, Overall_vis = compute_ablation(g_work, include_nl, **opts)
        g = g_work.copy()
        g["EQ_ab"] = EQ
        g["UD_ab"] = UD
        g["Overall_ab"] = Overall
        g["Overall_vis_ab"] = Overall_vis

        out_geo = os.path.join(ABL_DIR, f"{tag}_{name}.geojson")
        g.to_file(out_geo, driver="GeoJSON")
        base_eq = {k: v for k, v in (EQ_weights_250 if include_nl else EQ_weights_100).items() if k in g_work.columns}
        base_ud = {k: v for k, v in UD_weights.items() if k in g_work.columns}
        eq_w_used = renormalize(base_eq, opts.get("drop_eq")) if opts.get("drop_eq") in base_eq else base_eq
        ud_w_used = renormalize(base_ud, opts.get("drop_ud")) if opts.get("drop_ud") in base_ud else base_ud
        if opts.get("drop_overall") == "EQ":
            overall_w_used = {"EQ": 0.0, "UD": 1.0}
        elif opts.get("drop_overall") == "UD":
            overall_w_used = {"EQ": 1.0, "UD": 0.0}
        else:
            overall_w_used = OVERALL_weights

        save_map_png(g, "Overall_vis_ab", f"{tag}_{name}", eq_w_used, ud_w_used, overall_w_used)
        print("Wrote", out_geo)

# Fallback load if notebook state does not already have these
if "out_250_wgs" not in globals():
    out_250_wgs = gpd.read_file("lahore_index_250m.geojson")
if "out_100_wgs" not in globals():
    out_100_wgs = gpd.read_file("lahore_index_100m.geojson")

# run_ablations(out_250_wgs, include_nl=True, tag="250m")
run_ablations(out_100_wgs, include_nl=False, tag="100m")


Wrote ablations/100m_drop_EQ.geojson
Wrote ablations/100m_drop_UD.geojson
Wrote ablations/100m_drop_NDVI_sc.geojson
Wrote ablations/100m_drop_LST_sc.geojson


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_96524/2329546168.py:33: RuntimeWarning: invalid value encountered in divide
  out = np.where(denom == 0, np.nan, np.nansum(vals * w, axis=1) / denom)


Wrote ablations/100m_drop_AQI_sc.geojson
Wrote ablations/100m_drop_POI_access_sc.geojson
Wrote ablations/100m_drop_BFP_presence_sc.geojson


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_96524/2329546168.py:33: RuntimeWarning: invalid value encountered in divide
  out = np.where(denom == 0, np.nan, np.nansum(vals * w, axis=1) / denom)


Wrote ablations/100m_drop_Highrise_decay_sc.geojson
Wrote ablations/100m_drop_Height_sc.geojson
Wrote ablations/100m_drop_BFP_density_sc.geojson
Wrote ablations/100m_drop_Road_density_sc.geojson


## Run Ablations

To generate all ablation outputs:

1. Run all cells up to **Index Construction** and **Maps**.
2. Run the **Ablations** code cell at the end.

Outputs are written to:
- `notebooks/ablations/*.png` (maps)
- `notebooks/ablations/*.geojson` (grids)


In [9]:
# ==== WEIGHT-SCENARIO ABLATIONS (images only, no GeoJSON) ====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    ABL_W_DIR = BASE_DIR / "ablations_weight_scenarios"
else:
    ABL_W_DIR = BASE_DIR / "notebooks" / "ablations_weight_scenarios"
os.makedirs(ABL_W_DIR, exist_ok=True)

# Uses robust_0_10 from prior cells. Fallback for standalone execution:
if "robust_0_10" not in globals():
    def robust_0_10(series, invert=False, qlo=0.05, qhi=0.95):
        s = pd.Series(series).astype(float)
        lo, hi = s.quantile(qlo), s.quantile(qhi)
        denom = max(hi - lo, 1e-9)
        z = ((s - lo) / denom).clip(0, 1) * 10.0
        return (10 - z) if invert else z


def renorm(d):
    s = float(sum(d.values()))
    return {k: (v / s if s > 0 else 0.0) for k, v in d.items()}


def weighted_avg_safe(df, weights):
    cols = [c for c in weights if c in df.columns]
    if not cols:
        return np.full(len(df), np.nan)
    w = np.array([weights[c] for c in cols], dtype=float)
    x = df[cols].astype(float).values
    m = np.isnan(x)
    w_row = np.where(m, 0.0, w)
    den = w_row.sum(axis=1)
    out = np.where(den == 0, np.nan, np.nansum(x * w, axis=1) / den)
    return out


def save_weight_map(g, title, fname, eq_w, ud_w, ov_w):
    fig, ax = plt.subplots(figsize=(10, 10))
    g.plot(
        column="Overall_vis_ws",
        ax=ax,
        cmap="RdYlGn",
        linewidth=0.05,
        edgecolor="none",
        legend=True,
        legend_kwds={"shrink": 0.65, "fraction": 0.03, "pad": 0.01},
    )
    ax.set_title(title)
    ax.set_axis_off()

    def fmt(d):
        return "\n".join([f"{k.replace('_sc','')}: {v*100:.0f}%" for k, v in d.items()])

    txt = (
        "EQ weights\n" + fmt(eq_w) +
        "\n\nUD weights\n" + fmt(ud_w) +
        "\n\nOverall weights\n" + fmt(ov_w)
    )
    fig.text(
        0.015, 0.02, txt,
        ha="left", va="bottom", fontsize=8,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#666", "alpha": 0.9},
    )
    fig.savefig(str(ABL_W_DIR / fname), dpi=220, bbox_inches="tight")
    plt.close(fig)


def _key(weights):
    return tuple(sorted((k, round(v, 6)) for k, v in weights.items()))


def run_weight_scenarios(base_gdf, include_nl=False, tag="100m"):
    g0 = base_gdf.copy()

    if "POI_access_sc" not in g0.columns and "POI_n1_min" in g0.columns:
        g0["POI_access_sc"] = robust_0_10(g0["POI_n1_min"], invert=True)

    # Base weights (match index cell)
    eq_base = {"NDVI_sc": 0.35, "LST_sc": 0.35, "AQI_sc": 0.2, "POI_access_sc": 0.1}
    if include_nl:
        eq_base = {"NDVI_sc": 0.4, "LST_sc": 0.2, "NL_sc": 0.1, "AQI_sc": 0.2, "POI_access_sc": 0.1}

    ud_base = {
        "BFP_presence_sc": 0.1,
        "Highrise_decay_sc": 0.3,
        "Height_sc": 0.2,
        "BFP_density_sc": 0.2,
        "Road_density_sc": 0.2,
    }
    ov_base = {"EQ": 0.4, "UD": 0.6}

    # Keep only metrics available in this dataframe
    eq_metrics = [k for k in eq_base if k in g0.columns]
    ud_metrics = [k for k in ud_base if k in g0.columns]

    scenarios = []

    # 1) Base
    scenarios.append(("base", dict(eq_base), dict(ud_base), dict(ov_base)))

    # 2) Overall sweep (7 scenarios)
    for w_eq in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        scenarios.append((f"overall_eq_{int(w_eq*100)}", dict(eq_base), dict(ud_base), {"EQ": w_eq, "UD": 1-w_eq}))

    # 3) EQ single-metric heavy and light
    for m in eq_metrics:
        eq_h = {k: 0.1 for k in eq_metrics}
        eq_h[m] = 0.7
        scenarios.append((f"eq_heavy_{m}", eq_h, dict(ud_base), dict(ov_base)))

        eq_l = {k: 1.0 for k in eq_metrics}
        eq_l[m] = 0.05
        scenarios.append((f"eq_light_{m}", eq_l, dict(ud_base), dict(ov_base)))

    # 4) EQ drop-one (set zero then renorm)
    for m in eq_metrics:
        eq_d = {k: v for k, v in eq_base.items() if k != m}
        scenarios.append((f"eq_drop_{m}", eq_d, dict(ud_base), dict(ov_base)))

    # 5) UD single-metric heavy and light
    for m in ud_metrics:
        ud_h = {k: 0.1 for k in ud_metrics}
        ud_h[m] = 0.6
        scenarios.append((f"ud_heavy_{m}", dict(eq_base), ud_h, dict(ov_base)))

        ud_l = {k: 1.0 for k in ud_metrics}
        ud_l[m] = 0.05
        scenarios.append((f"ud_light_{m}", dict(eq_base), ud_l, dict(ov_base)))

    # 6) UD drop-one
    for m in ud_metrics:
        ud_d = {k: v for k, v in ud_base.items() if k != m}
        scenarios.append((f"ud_drop_{m}", dict(eq_base), ud_d, dict(ov_base)))

    # Deduplicate identical normalized setups
    unique = []
    seen = set()
    for name, eq_raw, ud_raw, ov_raw in scenarios:
        eq_w = renorm({k: v for k, v in eq_raw.items() if k in g0.columns})
        ud_w = renorm({k: v for k, v in ud_raw.items() if k in g0.columns})
        ov_w = renorm(ov_raw)
        key = (_key(eq_w), _key(ud_w), _key(ov_w))
        if key in seen:
            continue
        seen.add(key)
        unique.append((name, eq_w, ud_w, ov_w))

    print(f"Running {len(unique)} weight ablation scenarios...")

    for i, (name, eq_w, ud_w, ov_w) in enumerate(unique, start=1):
        EQ = weighted_avg_safe(g0, eq_w)
        UD = weighted_avg_safe(g0, ud_w)
        Overall = ov_w.get("EQ", 0.0) * EQ + ov_w.get("UD", 0.0) * UD

        g = g0.copy()
        g["Overall_ws"] = Overall
        g["Overall_vis_ws"] = robust_0_10(g["Overall_ws"], qlo=0.1, qhi=0.9)

        out_png = f"{tag}_weights_{name}.png"
        save_weight_map(g, f"{tag} weight scenario: {name}", out_png, eq_w, ud_w, ov_w)
        print(f"[{i}/{len(unique)}] Wrote", str(ABL_W_DIR / out_png))


# Fallback load
if "out_100_wgs" not in globals():
    out_100_wgs = gpd.read_file("lahore_index_100m.geojson")

run_weight_scenarios(out_100_wgs, include_nl=False, tag="100m")



Running 34 weight ablation scenarios...
[1/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_base.png
[2/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_overall_eq_20.png
[3/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_overall_eq_30.png
[4/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_overall_eq_50.png
[5/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_overall_eq_60.png
[6/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_overall_eq_70.png
[7/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_w

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_10576/2005597765.py:40: RuntimeWarning: invalid value encountered in divide
  out = np.where(den == 0, np.nan, np.nansum(x * w, axis=1) / den)


[18/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_eq_drop_AQI_sc.png
[19/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_eq_drop_POI_access_sc.png
[20/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_heavy_BFP_presence_sc.png
[21/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_light_BFP_presence_sc.png
[22/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_heavy_Highrise_decay_sc.png
[23/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_light_Highrise_decay_sc.png
[24/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_10576/2005597765.py:40: RuntimeWarning: invalid value encountered in divide
  out = np.where(den == 0, np.nan, np.nansum(x * w, axis=1) / den)


[31/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_drop_Highrise_decay_sc.png
[32/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_drop_Height_sc.png
[33/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_drop_BFP_density_sc.png
[34/34] Wrote /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/ablations_weight_scenarios/100m_weights_ud_drop_Road_density_sc.png
